In [1]:
!pip install pandas requests pyarrow openpyxl tqdm -q

In [2]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import quote
from tqdm import tqdm

# ============================================================
# NOAA PMEL ERDDAP RAMA / GTMBA Mooring Data
# ============================================================

BASE = "https://data.pmel.noaa.gov/pmel/erddap"

OUTPUT_DIR = Path("incois_omni_rama_validation_data")
RAW_DIR = OUTPUT_DIR / "raw_rama"
PROCESSED_DIR = OUTPUT_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = {
    "bob": {
        "name": "Bay_of_Bengal",
        "lat_min": 5,
        "lat_max": 22,
        "lon_min": 80,
        "lon_max": 98
    },
    "as": {
        "name": "Arabian_Sea",
        "lat_min": 5,
        "lat_max": 22,
        "lon_min": 60,
        "lon_max": 75
    }
}

START_TIME = "2020-01-01T00:00:00Z"
END_TIME = "2024-12-31T23:59:59Z"

# Main RAMA/TAO/PIRATA daily datasets useful for validation
DATASETS = {
    "wind_daily": "pmelTaoDyW",
    "barometric_pressure_daily": "pmelTaoDyBp",
    "sst_daily": "pmelTaoDySst",
    "air_temperature_daily": "pmelTaoDyAirt",
    "temperature_profile_daily": "pmelTaoDyT",
    "salinity_profile_daily": "pmelTaoDyS",
    "currents_daily": "pmelTaoDyCur",
    "rainfall_daily": "pmelTaoDyRain"
}


def get_erddap_variables(dataset_id):
    """
    Reads ERDDAP metadata and extracts available variable names.
    """
    info_url = f"{BASE}/info/{dataset_id}/index.csv"
    info = pd.read_csv(info_url)

    # ERDDAP info table usually contains Row Type and Variable Name columns
    info.columns = [c.strip() for c in info.columns]

    row_type_col = None
    var_name_col = None

    for col in info.columns:
        if col.lower() == "row type":
            row_type_col = col
        if col.lower() == "variable name":
            var_name_col = col

    if row_type_col is None or var_name_col is None:
        raise ValueError(f"Could not identify metadata columns for {dataset_id}. Columns: {info.columns.tolist()}")

    variables = info.loc[
        info[row_type_col].astype(str).str.lower() == "variable",
        var_name_col
    ].dropna().astype(str).tolist()

    return variables


def build_erddap_url(dataset_id, variables, region):
    """
    Builds ERDDAP tabledap URL with constraints.
    """
    columns = ",".join(variables)

    constraints = (
        f'&array="RAMA"'
        f'&time>={START_TIME}'
        f'&time<={END_TIME}'
        f'&latitude>={region["lat_min"]}'
        f'&latitude<={region["lat_max"]}'
        f'&longitude>={region["lon_min"]}'
        f'&longitude<={region["lon_max"]}'
    )

    query = columns + constraints

    encoded_query = quote(
        query,
        safe=",&=<>:\"-_.TZ"
    )

    return f"{BASE}/tabledap/{dataset_id}.csvp?{encoded_query}"


def download_one_dataset(label, dataset_id, region_key):
    region = REGIONS[region_key]

    print("\nDataset:", label)
    print("Dataset ID:", dataset_id)
    print("Region:", region["name"])

    try:
        variables = get_erddap_variables(dataset_id)
    except Exception as e:
        print("Failed to get variables:", e)
        return None

    print("Variables found:", variables)

    # Keep all columns because different PMEL products use different variable names.
    url = build_erddap_url(dataset_id, variables, region)

    output_csv = RAW_DIR / f"{label}_{region_key}_rama_2020_2024.csv"

    try:
        r = requests.get(url, timeout=600)
        r.raise_for_status()

        with open(output_csv, "wb") as f:
            f.write(r.content)

        df = pd.read_csv(output_csv)

        # Clean columns
        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
            .str.replace(r"\(|\)", "", regex=True)
            .str.replace("/", "_", regex=True)
        )

        df["study_region"] = region["name"]
        df["source_dataset"] = dataset_id
        df["source_label"] = label

        # Save cleaned CSV
        df.to_csv(output_csv, index=False)

        print("Downloaded:", output_csv)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())

        return df

    except Exception as e:
        print("Download failed:", label, dataset_id, region_key)
        print("Reason:", e)
        return None


all_frames = []

for label, dataset_id in DATASETS.items():
    for region_key in REGIONS.keys():
        df = download_one_dataset(label, dataset_id, region_key)
        if df is not None and len(df) > 0:
            all_frames.append(df)

print("\nCompleted RAMA data download.")
print("Non-empty datasets:", len(all_frames))


Dataset: wind_daily
Dataset ID: pmelTaoDyW
Region: Bay_of_Bengal
Variables found: ['array', 'station', 'wmo_platform_code', 'longitude', 'latitude', 'time', 'depth', 'WU_422', 'WV_423', 'WS_401', 'QWS_5401', 'SWS_6401', 'WD_410', 'QWD_5410', 'SWD_6410']
Downloaded: incois_omni_rama_validation_data/raw_rama/wind_daily_bob_rama_2020_2024.csv
Rows: 5481
Columns: ['array', 'station', 'wmo_platform_code', 'longitude_degrees_east', 'latitude_degrees_north', 'time_utc', 'depth_m', 'wu_422_m_s-1', 'wv_423_m_s-1', 'ws_401_m_s-1', 'qws_5401', 'sws_6401', 'wd_410_degrees_true', 'qwd_5410', 'swd_6410', 'study_region', 'source_dataset', 'source_label']

Dataset: wind_daily
Dataset ID: pmelTaoDyW
Region: Arabian_Sea
Variables found: ['array', 'station', 'wmo_platform_code', 'longitude', 'latitude', 'time', 'depth', 'WU_422', 'WV_423', 'WS_401', 'QWS_5401', 'SWS_6401', 'WD_410', 'QWD_5410', 'SWD_6410']
Downloaded: incois_omni_rama_validation_data/raw_rama/wind_daily_as_rama_2020_2024.csv
Rows: 1827


In [3]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("incois_omni_rama_validation_data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if len(all_frames) == 0:
    raise ValueError("No RAMA datasets downloaded. Check internet connection or ERDDAP constraints.")

rama_all = pd.concat(all_frames, ignore_index=True, sort=False)

combined_csv = PROCESSED_DIR / "rama_validation_all_bob_as_2020_2024.csv"
combined_parquet = PROCESSED_DIR / "rama_validation_all_bob_as_2020_2024.parquet"

rama_all.to_csv(combined_csv, index=False)
rama_all.to_parquet(combined_parquet, index=False)

print("Combined shape:", rama_all.shape)
print("Saved CSV:", combined_csv)
print("Saved Parquet:", combined_parquet)

rama_all.head()

Combined shape: (199065, 47)
Saved CSV: incois_omni_rama_validation_data/processed/rama_validation_all_bob_as_2020_2024.csv
Saved Parquet: incois_omni_rama_validation_data/processed/rama_validation_all_bob_as_2020_2024.parquet


,array,station,wmo_platform_code,longitude_degrees_east,latitude_degrees_north,time_utc,depth_m,wu_422_m_s-1,wv_423_m_s-1,ws_401_m_s-1,...,cd_310_degrees_true,qcs_5300,qcd_5310,scs_6300,cic_7300,rn_485_mm_hr,qrn_5485,srn_6485,rns_486_mm_hr,rnp_487_percent
0,RAMA,12n90e,23008,90.0,12.0,2020-01-01T12:00:00Z,-4.0,-6.7,-1.6,6.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RAMA,12n90e,23008,90.0,12.0,2020-01-02T12:00:00Z,-4.0,-6.6,-0.5,6.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RAMA,12n90e,23008,90.0,12.0,2020-01-03T12:00:00Z,-4.0,-5.7,-0.2,5.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RAMA,12n90e,23008,90.0,12.0,2020-01-04T12:00:00Z,-4.0,-3.6,-0.3,3.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RAMA,12n90e,23008,90.0,12.0,2020-01-05T12:00:00Z,-4.0,-4.5,-3.1,5.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
summary = (
    rama_all.groupby(["source_label", "source_dataset", "study_region"], as_index=False)
    .agg(
        records=("source_label", "count"),
        stations=("station", "nunique") if "station" in rama_all.columns else ("source_label", "count")
    )
)

summary_file = PROCESSED_DIR / "rama_download_summary.xlsx"
summary.to_excel(summary_file, index=False)

print("Saved summary:", summary_file)
summary

Saved summary: incois_omni_rama_validation_data/processed/rama_download_summary.xlsx


,source_label,source_dataset,study_region,records,stations
0,air_temperature_daily,pmelTaoDyAirt,Arabian_Sea,2121,2
1,air_temperature_daily,pmelTaoDyAirt,Bay_of_Bengal,5481,3
2,barometric_pressure_daily,pmelTaoDyBp,Arabian_Sea,294,1
3,barometric_pressure_daily,pmelTaoDyBp,Bay_of_Bengal,1827,1
4,currents_daily,pmelTaoDyCur,Arabian_Sea,294,1
5,currents_daily,pmelTaoDyCur,Bay_of_Bengal,3742,2
6,rainfall_daily,pmelTaoDyRain,Arabian_Sea,1913,2
7,rainfall_daily,pmelTaoDyRain,Bay_of_Bengal,5481,3
8,salinity_profile_daily,pmelTaoDyS,Arabian_Sea,15141,2
9,salinity_profile_daily,pmelTaoDyS,Bay_of_Bengal,40194,3


In [5]:
import shutil
from google.colab import files

zip_name = "incois_omni_rama_validation_data"

shutil.make_archive(zip_name, "zip", "incois_omni_rama_validation_data")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>